# 05 - Visualisation des tables RSNA et qualité d'image

Ce notebook permet de visualiser la base de donnée avec les images du dataset rnsa `data/database.sqlite`.
* On afficher les tables `cases/runs/evaluations/prompts`
* On évalue la qualité d'image réelle (luminosité, contraste, densité de contours) pour la comparer au dataset prlote (dataset de test de Kaggle)

Pour visualiser le dataset de test (`fkarimovv/abnormal-lung`) voir notebook `04_Database_visualisation.ipynb`.

Pre-requis avant d'executer ce notebook :
1. `python scripts/import_rsna_pneumonia.py --max-cases 100` (necessite un compte Kaggle authentifie ayant accepte les regles de la competition `rsna-pneumonia-detection-challenge`)
2. `python scripts/setup_db.py --all` (cree les tables et charge `data/cases.csv` + les prompts)
3. Lancer l'evaluation MedGemma sur ces cas via `scripts/run_prompt_evaluation.py`

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))
from src.database import get_runs, get_evaluations, get_cases, get_prompts
import pandas as pd

DB_PATH = Path('..') / 'data' / 'database.sqlite'
BRUTES_RSA = Path('..') / 'data' / 'brutes_rsa'

### Images disponibles

In [2]:
total = len(list(BRUTES_RSA.glob('*.png'))) if BRUTES_RSA.exists() else 0
print(f"Images RSNA disponibles dans {BRUTES_RSA} : {total}")

Images RSNA disponibles dans ..\data\brutes_rsa : 26684


### `cases` table

In [3]:
cases_df = pd.DataFrame(get_cases(DB_PATH))
display(cases_df)
cases_df['ground_truth_label'].value_counts()

,id,image_path,source,ground_truth_label,split,notes
0,1,data/brutes_rsa/rsna_pneumonia_normal_00001.png,rsna-pneumonia-detection-challenge,normal,external,Imported from RSNA Pneumonia Detection Challenge
1,2,data/brutes_rsa/rsna_pneumonia_normal_00002.png,rsna-pneumonia-detection-challenge,normal,external,Imported from RSNA Pneumonia Detection Challenge
2,3,data/brutes_rsa/rsna_pneumonia_suspected_opaci...,rsna-pneumonia-detection-challenge,suspected_opacity,external,Imported from RSNA Pneumonia Detection Challenge
3,4,data/brutes_rsa/rsna_pneumonia_suspected_opaci...,rsna-pneumonia-detection-challenge,suspected_opacity,external,Imported from RSNA Pneumonia Detection Challenge
4,5,data/brutes_rsa/rsna_pneumonia_suspected_opaci...,rsna-pneumonia-detection-challenge,suspected_opacity,external,Imported from RSNA Pneumonia Detection Challenge
...,...,...,...,...,...,...
26679,26680,data/brutes_rsa/rsna_pneumonia_suspected_opaci...,rsna-pneumonia-detection-challenge,suspected_opacity,external,Imported from RSNA Pneumonia Detection Challenge
26680,26681,data/brutes_rsa/rsna_pneumonia_normal_26681.png,rsna-pneumonia-detection-challenge,normal,external,Imported from RSNA Pneumonia Detection Challenge
26681,26682,data/brutes_rsa/rsna_pneumonia_normal_26682.png,rsna-pneumonia-detection-challenge,normal,external,Imported from RSNA Pneumonia Detection Challenge
26682,26683,data/brutes_rsa/rsna_pneumonia_normal_26683.png,rsna-pneumonia-detection-challenge,normal,external,Imported from RSNA Pneumonia Detection Challenge


ground_truth_label
normal               20672
suspected_opacity     6012
Name: count, dtype: int64

### `runs` table

In [4]:
display(pd.DataFrame(get_runs(DB_PATH)))

,id,case_id,prompt_id,image_path,model_name,prediction_json,predicted_class,confidence,latency_ms,created_at
0,470,26682,1,C:\Users\auber.MSI\Projets\TD-Mastercamp\Solut...,medgemma-4b-it,"{""image_quality"": ""good"", ""predicted_class"": ""...",suspected_opacity,0.60,18744,2026-07-05 23:24:56
1,469,26583,1,C:\Users\auber.MSI\Projets\TD-Mastercamp\Solut...,medgemma-4b-it,"{""image_quality"": ""good"", ""predicted_class"": ""...",uncertain,0.20,17808,2026-07-05 23:24:37
2,468,26423,1,C:\Users\auber.MSI\Projets\TD-Mastercamp\Solut...,medgemma-4b-it,"{""image_quality"": ""good"", ""predicted_class"": ""...",suspected_opacity,0.30,16099,2026-07-05 23:24:19
3,467,25615,1,C:\Users\auber.MSI\Projets\TD-Mastercamp\Solut...,medgemma-4b-it,"{""image_quality"": ""good"", ""predicted_class"": ""...",uncertain,0.00,17974,2026-07-05 23:24:03
4,466,25018,1,C:\Users\auber.MSI\Projets\TD-Mastercamp\Solut...,medgemma-4b-it,"{""image_quality"": ""good"", ""predicted_class"": ""...",normal,0.95,21886,2026-07-05 23:23:44
...,...,...,...,...,...,...,...,...,...,...
454,5,21031,5,C:\Users\auber.MSI\Projets\TD-Mastercamp\Solut...,medgemma-4b-it,"{""image_quality"": ""good"", ""predicted_class"": ""...",suspected_opacity,0.80,12874,2026-07-05 16:38:18
455,4,1781,5,C:\Users\auber.MSI\Projets\TD-Mastercamp\Solut...,medgemma-4b-it,"{""image_quality"": ""good"", ""predicted_class"": ""...",suspected_opacity,0.80,14001,2026-07-05 16:38:05
456,3,9677,5,C:\Users\auber.MSI\Projets\TD-Mastercamp\Solut...,medgemma-4b-it,"{""image_quality"": ""good"", ""predicted_class"": ""...",uncertain,0.00,22538,2026-07-05 16:37:51
457,2,18089,5,C:\Users\auber.MSI\Projets\TD-Mastercamp\Solut...,medgemma-4b-it,"{""image_quality"": ""good"", ""predicted_class"": ""...",suspected_opacity,0.75,14381,2026-07-05 16:37:28


### `evaluations` table (jointure runs + evaluations)

In [5]:
eval_df = pd.DataFrame(get_evaluations(DB_PATH))
eval_df

,case_id,run_id,evaluation_id,prompt_id,prompt_name,prompt_version,ground_truth_label,predicted_class,confidence,latency
0,1079,371,371,1,baseline,v0,suspected_opacity,uncertain,0.00,21173
1,1520,372,372,1,baseline,v0,suspected_opacity,uncertain,0.00,20572
2,1656,373,373,1,baseline,v0,suspected_opacity,uncertain,0.00,20988
3,1762,374,374,1,baseline,v0,normal,uncertain,0.00,20526
4,1832,375,375,1,baseline,v0,suspected_opacity,uncertain,0.00,14143
...,...,...,...,...,...,...,...,...,...,...
454,23568,207,207,9,improved_aux,v4,normal,suspected_opacity,0.53,11390
455,24954,208,208,9,improved_aux,v4,suspected_opacity,suspected_opacity,0.75,11160
456,25130,175,175,9,improved_aux,v4,normal,normal,0.75,21093
457,25249,209,209,9,improved_aux,v4,normal,normal,0.75,13171


### `prompts` table

In [6]:
display(pd.DataFrame(get_prompts(DB_PATH)))

,id,prompt_name,prompt_version,prompt_text,created_at
0,13,improved,v11,Classify this frontal chest X-ray.\n\nReturn o...,2026-07-05 20:20:26
1,12,improved,v10,You are an educational radiology assistant for...,2026-07-05 20:17:21
2,11,improved,v9,You are an educational radiology assistant for...,2026-07-05 20:14:30
3,10,improved,v8,You are an educational radiology assistant for...,2026-07-05 19:28:27
4,9,improved_aux,v4,You are an educational radiology assistant for...,2026-07-05 18:54:04
5,8,improved,v7,You are an educational radiology assistant for...,2026-07-05 17:54:14
6,7,improved,v6,You are an educational radiology assistant for...,2026-07-05 17:35:22
7,1,baseline,v0,You are an educational radiology assistant for...,2026-07-05 12:11:11
8,2,improved,v1,You are an educational radiology assistant for...,2026-07-05 12:11:11
9,3,improved,v2,You are an educational radiology assistant for...,2026-07-05 12:11:11


## Qualité d'image : Kaggle vs RSNA

Le champ `image_quality` renvoyé dans le JSON (`good`/`not good`) vient de `src/preprocessing.basic_quality_flag`. C'est un **placeholder factice** qui regarde juste si "uncertain" ou "limited" apparaît dans le *nom de fichier*, pas une vraie analyse d'image. Les fichiers RSNA (`rsna_pneumonia_{label}_{index}.png`) ne contiennent jamais ces mots, donc ce champ vaut toujours "good" pour RSNA, quelle que soit la qualité réelle. On calcule ici de vraies statistiques d'image (luminosité, contraste, densité de contours, taille) pour comparer objectivement les deux datasets.

In [7]:
import random
import numpy as np
from PIL import Image, ImageStat, ImageFilter

def image_stats(path):
    img = Image.open(path).convert('L')
    w, h = Image.open(path).size
    stat = ImageStat.Stat(img)
    edges = np.asarray(img.filter(ImageFilter.FIND_EDGES), dtype=np.float32) / 255.0
    return {
        'brightness': stat.mean[0],
        'contrast': stat.stddev[0],
        'edge_density': float(edges.mean()),
        'width': w,
        'height': h,
    }

random.seed(1)
kaggle_files = random.sample(list((Path('..') / 'data' / 'brutes_kaggle').glob('*.jpg')), 200)
rsna_files = random.sample(list((Path('..') / 'data' / 'brutes_rsa').glob('*.png')), 200)

for name, files in (('Kaggle', kaggle_files), ('RSNA', rsna_files)):
    rows = [image_stats(f) for f in files]
    df_stats = pd.DataFrame(rows)
    print(f'=== {name} (n={len(rows)}) ===')
    print(df_stats[['brightness', 'contrast', 'edge_density', 'width', 'height']].describe().loc[['mean', 'std']])
    print()

=== Kaggle (n=200) ===
      brightness   contrast  edge_density        width       height
mean  124.918435  40.247842      0.019863  2398.280000  2319.695000
std    13.483479   6.000839      0.003170   208.097367   241.864724

=== RSNA (n=200) ===
      brightness   contrast  edge_density   width  height
mean  129.602682  60.435347      0.018291  1024.0  1024.0
std    24.287086   9.744680      0.003924     0.0     0.0



**Observation (mesurée sur 200 images de chaque dataset)** : le contraste RSNA est nettement plus élevé et variable (~60 ± 10) que Kaggle (~40 ± 6), et la luminosité RSNA est presque deux fois plus variable (écart-type ~24 vs ~13). Autre différence : toutes les images RSNA font exactement 1024×1024 (le challenge standardise la taille), alors que Kaggle varie en résolution.

Un contraste plus élevé et plus variable est cohérent avec des radios cliniques réelles (expositions différentes selon le contexte, portable vs debout) plutôt qu'un dataset homogène. Cela appuie l'hypothèse que les images RSNA sont visuellement plus hétérogènes, indépendamment du contenu médical.

### Est-ce que le contraste/la luminosité explique les erreurs de MedGemma sur RSNA ?

Même approche que dans le notebook 02 pour Kaggle : on compare les statistiques d'image entre les cas bien classifiés et les cas en erreur, sur les cas RSNA déjà évalués par MedGemma.

In [10]:
# mapping case_id -> chemin fichier, a partir de la table cases deja chargee
id_to_path = dict(zip(cases_df['id'], cases_df['image_path']))

eval_df['correct'] = eval_df['ground_truth_label'] == eval_df['predicted_class']

rows = []
for _, r in eval_df.iterrows():
    path = Path('..') / id_to_path[r['case_id']]
    if not path.exists():
        continue
    stats = image_stats(path)
    stats['case_id'] = r['case_id']
    stats['correct'] = r['correct']
    stats['prompt_name'] = r['prompt_name']
    stats['prompt_version'] = r['prompt_version']
    rows.append(stats)

quality_df = pd.DataFrame(rows)
quality_df.groupby('correct')[['brightness', 'contrast', 'edge_density']].agg(['mean', 'std'])

brightness              contrast            edge_density          
               mean        std       mean        std         mean       std
correct                                                                    
False    123.657958  23.933893  59.130932   9.507667     0.017346  0.004057
True     127.997240  25.532081  59.144162  10.605138     0.017177  0.004386

**Conclusion** : aucune différence notable entre cas corrects et erreurs sur ces statistiques globales (contraste, luminosité, densité de contours quasi identiques). Comme sur Kaggle, une mauvaise qualité d'image globale n'explique pas les erreurs de MedGemma sur RSNA le problème est localisé à des éléments spécifiques de l'image (matériel médical, tissu superposé), pas une question de qualité générale mesurable simplement. Ça confirme qu'il ne s'agit pas d'un simple biais d'exposition ou de contraste, mais bien d'un problème de contenu visuel spécifique.